In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("C:\\Users\\RAHUL\\OneDrive\\Documents\\Desktop\\Projects\\EDA_projects\\retail-sales-profitability\\data\\raw\\Sample - Superstore.csv", encoding="windows-1252")

df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [3]:
print("Shape of the dataset is: ", df.shape)
print("No. of rows in the dataset is: ", df.shape[0])
print("No. of columns in the dataset is: ", df.shape[1])

Shape of the dataset is:  (9994, 21)
No. of rows in the dataset is:  9994
No. of columns in the dataset is:  21


In [4]:
df.dtypes.to_frame('Data Types')

,Data Types
Row ID,int64
Order ID,str
Order Date,str
Ship Date,str
Ship Mode,str
Customer ID,str
Customer Name,str
Segment,str
Country,str
City,str


In [5]:
df.isnull().sum().to_frame("Null Values Count")

,Null Values Count
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Customer Name,0
Segment,0
Country,0
City,0


In [6]:
df.duplicated().sum()

np.int64(0)

### Data Profiling and Summary Statistics.

In [7]:
print("Dataset Information")
df.info()

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18 

In [8]:
print("Numerical Summary")
df[["Sales", "Quantity", "Discount", "Profit"]].describe().T

Numerical Summary


,count,mean,std,min,25%,50%,75%,max
Sales,9994.0,229.858001,623.245101,0.444,17.28000,54.4900,209.940,22638.480
Quantity,9994.0,3.789574,2.225110,1.000,2.00000,3.0000,5.000,14.000
Discount,9994.0,0.156203,0.206452,0.000,0.00000,0.2000,0.200,0.800
Profit,9994.0,28.656896,234.260108,-6599.978,1.72875,8.6665,29.364,8399.976


In [9]:
print("Unique Values in Each Column")
df.nunique().to_frame("Unique Values Count")

Unique Values in Each Column


,Unique Values Count
Row ID,9994
Order ID,5009
Order Date,1237
Ship Date,1334
Ship Mode,4
Customer ID,793
Customer Name,793
Segment,3
Country,1
City,531


### Business Rule Validation

In [10]:
order_dates = pd.to_datetime(df["Order Date"], format="%m/%d/%Y", errors="coerce")

ship_dates = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y", errors="coerce")

In [11]:
checks = {"Invalid Order Dates": order_dates.isna().sum(),
        "Invalid Ship Dates": ship_dates.isna().sum(),
        "Ship Dates Before Order Dates": (ship_dates < order_dates).sum(),
        "Sales Less Than or Equal to Zero": (df["Sales"] <= 0).sum(),
        "Quantity Less Than or Equal to Zero": (df["Quantity"] <= 0).sum(),
        "Discount Outside 0 to 1": (~df["Discount"].between(0, 1)).sum(),
        "Rows With Negative Profit": (df["Profit"] < 0).sum()}

print("Earliest Order Date:", order_dates.min())
print("Latest Order Date:", order_dates.max())

pd.Series(checks).to_frame("Count")

Earliest Order Date: 2014-01-03 00:00:00
Latest Order Date: 2017-12-30 00:00:00


,Count
Invalid Order Dates,0
Invalid Ship Dates,0
Ship Dates Before Order Dates,0
Sales Less Than or Equal to Zero,0
Quantity Less Than or Equal to Zero,0
Discount Outside 0 to 1,0
Rows With Negative Profit,1871


### Loss-Making Sales Analysis

In [12]:
loss_rows = df[df["Profit"] < 0]

print("Loss-making rows:", len(loss_rows))

print("Percentage of loss-making rows:", round(len(loss_rows) / len(df) * 100, 2), "%")

(df.assign(Loss_Making_Row=df["Profit"] < 0)
    .groupby("Sub-Category")
    .agg(Total_Sales=("Sales", "sum"),
         Total_Profit=("Profit", "sum"),
         Loss_Making_Rows=("Loss_Making_Row", "sum"),
         Average_Discount=("Discount", "mean"))
    .sort_values("Total_Profit")
)


Loss-making rows: 1871
Percentage of loss-making rows: 18.72 %


,Total_Sales,Total_Profit,Loss_Making_Rows,Average_Discount
Sub-Category,,,,
Tables,206965.5320,-17725.4811,203,0.261285
Bookcases,114879.9963,-3472.5560,109,0.211140
Supplies,46673.5380,-1189.0995,33,0.076842
Fasteners,3024.2800,949.5182,12,0.082028
Machines,189238.6310,3384.7569,44,0.306087
Labels,12486.3120,5546.2540,0,0.068681
Art,27118.7920,6527.7870,0,0.074874
Envelopes,16476.4020,6964.1767,0,0.080315
Furnishings,91705.1640,13059.1436,167,0.138349


### Discount and Profitability Analysis

In [13]:
discount_summary = (
    df.groupby("Discount")
    .agg(
        Rows=("Profit", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Profit=("Profit", "mean"),
        Loss_Making_Rows=("Profit", lambda x: (x < 0).sum())
        )
)

discount_summary["Loss_Making_Percentage"] = discount_summary["Loss_Making_Rows"] / discount_summary["Rows"] * 100

discount_summary.round(2).sort_index()

,Rows,Total_Sales,Total_Profit,Average_Profit,Loss_Making_Rows,Loss_Making_Percentage
Discount,,,,,,
0.00,4798,1087908.47,320987.60,66.90,0,0.00
0.10,94,54369.35,9029.18,96.06,4,4.26
0.15,52,27558.52,1418.99,27.29,17,32.69
0.20,3657,764594.37,90337.31,24.70,502,13.73
0.30,227,103226.66,-10369.28,-45.68,208,91.63
0.32,27,14493.46,-2391.14,-88.56,27,100.00
0.40,206,116417.78,-23057.05,-111.93,180,87.38
0.45,11,5484.97,-2493.11,-226.65,11,100.00
0.50,66,58918.54,-20506.43,-310.70,66,100.00


### Discount Impact Across Product Sub-Categories.

In [14]:
discount_comparison = (
df.assign(
        Discount_Band=df["Discount"].ge(0.30).map({
            False: "Below 30%",
            True: "30% and Above"
        })
    )
    .groupby(["Sub-Category", "Discount_Band"])
    .agg(
        Rows=("Profit", "size"),
        Total_Sales=("Sales", "sum"),
        Total_Profit=("Profit", "sum"),
        Average_Profit=("Profit", "mean"),
        Loss_Rows=("Profit", lambda x: (x < 0).sum())
    )
    .reset_index()
)

discount_comparison["Loss_Rate_%"] = (
    discount_comparison["Loss_Rows"]
    / discount_comparison["Rows"]
    * 100
)

display(
    discount_comparison
    .round(2)
    .sort_values(["Sub-Category", "Discount_Band"])
)

,Sub-Category,Discount_Band,Rows,Total_Sales,Total_Profit,Average_Profit,Loss_Rows,Loss_Rate_%
0,Accessories,Below 30%,775,167380.32,41936.64,54.11,91,11.74
1,Appliances,30% and Above,67,3382.53,-8629.64,-128.80,67,100.00
2,Appliances,Below 30%,399,104149.63,26767.65,67.09,0,0.00
3,Art,Below 30%,796,27118.79,6527.79,8.20,0,0.00
4,Binders,30% and Above,613,36140.61,-38510.50,-62.82,613,100.00
5,Binders,Below 30%,910,167272.12,68732.26,75.53,0,0.00
6,Bookcases,30% and Above,70,28544.00,-11097.76,-158.54,69,98.57
7,Bookcases,Below 30%,158,86336.00,7625.21,48.26,40,25.32
8,Chairs,30% and Above,158,70005.50,-6737.12,-42.64,148,93.67
9,Chairs,Below 30%,459,258443.60,33327.28,72.61,87,18.95


### Initial Data Type Corrections.

In [15]:
df_clean = df.copy()

df_clean["Order Date"] = pd.to_datetime(
    df_clean["Order Date"],
    format="%m/%d/%Y"
)

df_clean["Ship Date"] = pd.to_datetime(
    df_clean["Ship Date"],
    format="%m/%d/%Y"
)

print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

display(
    df_clean[["Order Date", "Ship Date"]]
    .dtypes
    .to_frame("Data Type")
)

print("Missing values after date conversion:")
display(
    df_clean[["Order Date", "Ship Date"]]
    .isna()
    .sum()
    .to_frame("Missing Values")
)

Original dataset shape: (9994, 21)
Working dataset shape: (9994, 21)


,Data Type
Order Date,datetime64[us]
Ship Date,datetime64[us]


Missing values after date conversion:


,Missing Values
Order Date,0
Ship Date,0


### Postal Code Validation.

In [16]:
print("Postal Code data type:", df_clean["Postal Code"].dtype)

print("Unique postal codes:", df_clean["Postal Code"].nunique())

print("Postal codes below 10000:", (df_clean["Postal Code"] < 10000).sum())

print("Postal codes equal to zero:", (df_clean["Postal Code"] == 0).sum())

display(
    df_clean.loc[df_clean["Postal Code"] < 10000, ["Postal Code", "City", "State"]]
    .drop_duplicates()
    .sort_values("Postal Code")
    .head(20)
)

Postal Code data type: int64
Unique postal codes: 631
Postal codes below 10000: 449
Postal codes equal to zero: 0


,Postal Code,City,State
2415,1040,Holyoke,Massachusetts
2028,1453,Leominster,Massachusetts
9027,1752,Marlborough,Massachusetts
8331,1810,Andover,Massachusetts
422,1841,Lawrence,Massachusetts
346,1852,Lowell,Massachusetts
6295,1915,Beverly,Massachusetts
377,2038,Franklin,Massachusetts
6267,2138,Cambridge,Massachusetts
2393,2148,Malden,Massachusetts


### Postal Code Standardization.

In [17]:
short_zip_mask = df_clean["Postal Code"] < 10000

df_clean["Postal Code"] = df_clean["Postal Code"].astype(str).str.zfill(5)

print("Postal Code data type:", df_clean["Postal Code"].dtype)

print("Postal codes not having 5 characters:", df_clean["Postal Code"].str.len().ne(5).sum())

print("Dataset shape after correction:", df_clean.shape)

display(
    df_clean.loc[short_zip_mask, ["Postal Code", "City", "State"]]
    .drop_duplicates()
    .head(20)
)

Postal Code data type: str
Postal codes not having 5 characters: 0
Dataset shape after correction: (9994, 21)


,Postal Code,City,State
185,06824,Fairfield,Connecticut
197,07090,Westfield,New Jersey
267,07960,Morristown,New Jersey
298,07109,Belleville,New Jersey
306,08701,Lakewood,New Jersey
313,07601,Hackensack,New Jersey
346,01852,Lowell,Massachusetts
366,06040,Manchester,Connecticut
377,02038,Franklin,Massachusetts
395,02886,Warwick,Rhode Island


### Record Granularity and Identifier Validation.

In [18]:
print("Total rows:", len(df_clean))

print("Unique Row IDs:", df_clean["Row ID"].nunique())

print("Duplicate Row IDs:", df_clean["Row ID"].duplicated().sum())

print("Unique Orders:", df_clean["Order ID"].nunique())

print("Orders with multiple rows:", (df_clean.groupby("Order ID").size() > 1).sum())

order_size_summary = (
    df_clean.groupby("Order ID")
    .size()
    .describe()
    .to_frame("Rows per Order")
)

display(order_size_summary)

print("Repeated Order ID and Product ID combinations:", df_clean.duplicated(subset=["Order ID", "Product ID"]).sum())

Total rows: 9994
Unique Row IDs: 9994
Duplicate Row IDs: 0
Unique Orders: 5009
Orders with multiple rows: 2471


,Rows per Order
count,5009.000000
mean,1.995209
std,1.410671
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,14.000000


Repeated Order ID and Product ID combinations: 8


### Repeated Order-Product Investigation.

In [19]:
repeated_combinations = df_clean[
    df_clean.duplicated(
        subset=["Order ID", "Product ID"],
        keep=False
    )
].sort_values(["Order ID", "Product ID", "Row ID"])

display(
    repeated_combinations[
        [
            "Row ID",
            "Order ID",
            "Order Date",
            "Customer Name",
            "Product ID",
            "Product Name",
            "Sales",
            "Quantity",
            "Discount",
            "Profit"
        ]
    ]
)

print("Number of rows involved:", len(repeated_combinations))

,Row ID,Order ID,Order Date,Customer Name,Product ID,Product Name,Sales,Quantity,Discount,Profit
6498,6499,CA-2015-103135,2015-07-24,Shirley Schmidt,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",135.090,9,0.0,62.1414
6500,6501,CA-2015-103135,2015-07-24,Shirley Schmidt,OFF-BI-10000069,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",90.060,6,0.0,41.4276
350,351,CA-2016-129714,2016-09-01,Adam Bellavance,OFF-PA-10001970,Xerox 1881,24.560,2,0.0,11.5432
352,353,CA-2016-129714,2016-09-01,Adam Bellavance,OFF-PA-10001970,Xerox 1881,49.120,4,0.0,23.0864
1300,1301,CA-2016-137043,2016-12-23,Logan Currie,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",572.760,6,0.0,166.1004
1301,1302,CA-2016-137043,2016-12-23,Logan Currie,FUR-FU-10003664,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",286.380,3,0.0,83.0502
9168,9169,CA-2016-140571,2016-03-15,Sanjit Jacobs,OFF-PA-10001954,Xerox 1964,319.760,14,0.0,147.0896
9169,9170,CA-2016-140571,2016-03-15,Sanjit Jacobs,OFF-PA-10001954,Xerox 1964,45.680,2,0.0,21.0128
7881,7882,CA-2017-118017,2017-12-03,Lena Cacioppo,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,76.752,6,0.2,10.5534
7882,7883,CA-2017-118017,2017-12-03,Lena Cacioppo,TEC-AC-10002006,Memorex Micro Travel Drive 16 GB,102.336,8,0.2,14.0712


Number of rows involved: 16


### Exact Duplicate Record Validation.

In [20]:
business_columns = df_clean.columns.drop("Row ID")

exact_duplicate_mask = df_clean.duplicated(subset=business_columns, keep=False)

exact_duplicate_rows = df_clean.loc[exact_duplicate_mask].sort_values(["Order ID", "Product ID", "Row ID"])

print("Rows matching another record across all columns except Row ID:", exact_duplicate_mask.sum())

print("Extra repeated records after keeping one copy:", df_clean.duplicated(subset=business_columns).sum())

display(exact_duplicate_rows)

Rows matching another record across all columns except Row ID: 2
Extra repeated records after keeping one copy: 1


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
3405,3406,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588
3406,3407,US-2014-150119,2014-04-23,2014-04-27,Standard Class,LB-16795,Laurel Beltran,Home Office,United States,Columbus,...,43229,East,FUR-CH-10002965,Furniture,Chairs,Global Leather Highback Executive Chair with P...,281.372,2,0.3,-12.0588


### Categorical Data Profiling.

In [21]:
categorical_columns = [
    "Ship Mode",
    "Segment",
    "Country",
    "Region",
    "Category",
    "Sub-Category"
]

for column in categorical_columns:
    print(f"\nDistribution of {column}")
    display(
        df[column]
        .value_counts(dropna=False)
        .to_frame("Count")
    )

print("\nTop 10 Cities by Number of Records")
display(
    df["City"]
    .value_counts()
    .head(10)
    .to_frame("Count")
)

print("\nTop 10 States by Number of Records")
display(
    df["State"]
    .value_counts()
    .head(10)
    .to_frame("Count")
)


Distribution of Ship Mode


,Count
Ship Mode,
Standard Class,5968
Second Class,1945
First Class,1538
Same Day,543



Distribution of Segment


,Count
Segment,
Consumer,5191
Corporate,3020
Home Office,1783



Distribution of Country


...

,Count
Sub-Category,
Binders,1523
Paper,1370
Furnishings,957
Phones,889
Storage,846
Art,796
Accessories,775
Chairs,617
Appliances,466



Top 10 Cities by Number of Records


,Count
City,
New York City,915
Los Angeles,747
Philadelphia,537
San Francisco,510
Seattle,428
Houston,377
Chicago,314
Columbus,222
San Diego,170



Top 10 States by Number of Records


,Count
State,
California,2001
New York,1128
Texas,985
Pennsylvania,587
Washington,506
Illinois,492
Ohio,469
Florida,383
Michigan,255
